In [16]:
import pandas as pd
import numpy as np
import networkx as nx
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

print("All imports successful!")

All imports successful!


In [6]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
edges_df = pd.read_csv('txs_edgelist.csv')

# Keep only labeled transactions (filter out 'unknown' or unlabelled rows)
train_labeled = train_df[train_df['label'].isin(['0', '1', 0, 1])].copy()
train_labeled['label'] = train_labeled['label'].astype(int)

print(f"Train labeled shape: {train_labeled.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Class distribution in train:\n{train_labeled['label'].value_counts(normalize=True)}")

Train labeled shape: (31235, 168)
Test shape: (15329, 168)
Class distribution in train:
label
0    0.883336
1    0.116664
Name: proportion, dtype: float64


In [21]:
print("Calculating streamlined topology, top-30 1-hop neighbor stats, and neighbor label propagation...")

# 1. Graph Topology Metrics
G = nx.from_pandas_edgelist(edges_df, source='txId1', target='txId2', create_using=nx.DiGraph())
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
pagerank = nx.pagerank(G, alpha=0.85)

# 2. Extract 1-Hop Neighbor Statistics for Top 30 Local Features (reduces high-dimensional noise)
feat_cols = [f'feat_{i}' for i in range(1, 31)]
all_nodes_df = pd.concat([train_df, test_df], ignore_index=True)
feature_lookup = all_nodes_df.set_index('txId')[feat_cols]

# Incoming & Outgoing 1-Hop Means
in_edges = edges_df.merge(feature_lookup, left_on='txId1', right_index=True)
in_neighbor_means = in_edges.groupby('txId2')[feat_cols].mean().add_prefix('in_neigh_mean_')

out_edges = edges_df.merge(feature_lookup, left_on='txId2', right_index=True)
out_neighbor_means = out_edges.groupby('txId1')[feat_cols].mean().add_prefix('out_neigh_mean_')

# 3. Semi-Supervised Neighbor Label Propagation
known_labels = train_labeled.set_index('txId')['label'].to_dict()
edges_with_labels = edges_df.copy()
edges_with_labels['source_label'] = edges_with_labels['txId1'].map(known_labels)
edges_with_labels['target_label'] = edges_with_labels['txId2'].map(known_labels)

# Calculate ratio of known illicit transactions in direct incoming/outgoing edges
in_illicit_ratio = edges_with_labels.groupby('txId2')['source_label'].mean().rename('in_neigh_illicit_ratio')
out_illicit_ratio = edges_with_labels.groupby('txId1')['target_label'].mean().rename('out_neigh_illicit_ratio')

# 4. Master Feature Generation Function
def create_streamlined_features(df):
    df = df.copy()
    
    # Topology
    df['in_degree'] = df['txId'].map(in_degrees).fillna(0)
    df['out_degree'] = df['txId'].map(out_degrees).fillna(0)
    df['total_degree'] = df['in_degree'] + df['out_degree']
    df['degree_ratio'] = (df['in_degree'] + 1) / (df['out_degree'] + 1)
    df['degree_imbalance'] = (df['in_degree'] - df['out_degree']) / (df['total_degree'] + 1)
    df['pagerank'] = df['txId'].map(pagerank).fillna(0)
    
    # Selected 1-Hop Aggregations
    df = df.merge(in_neighbor_means, left_on='txId', right_index=True, how='left')
    df = df.merge(out_neighbor_means, left_on='txId', right_index=True, how='left')
    
    # Label propagation features (-1 indicates no labeled neighbors)
    df = df.merge(in_illicit_ratio, left_on='txId', right_index=True, how='left')
    df = df.merge(out_illicit_ratio, left_on='txId', right_index=True, how='left')
    
    # Fill label ratios with -1, remaining missing numeric features with 0
    df['in_neigh_illicit_ratio'] = df['in_neigh_illicit_ratio'].fillna(-1)
    df['out_neigh_illicit_ratio'] = df['out_neigh_illicit_ratio'].fillna(-1)
    
    # Top 5 Local vs Neighbor Differences
    for i in range(1, 6):
        col = f'feat_{i}'
        if f'in_neigh_mean_{col}' in df.columns:
            df[f'{col}_diff_in_neigh'] = df[col] - df[f'in_neigh_mean_{col}']
            
    return df.fillna(0)

# Build feature datasets
train_features = create_streamlined_features(train_labeled)
test_features = create_streamlined_features(test_df)

print(f"Streamlined total engineered features: {train_features.shape[1] - 4}")

Calculating streamlined topology, top-30 1-hop neighbor stats, and neighbor label propagation...
Streamlined total engineered features: 237


In [22]:
ignore_cols = ['index', 'txId', 'label', 'time_step']
feature_cols = [c for c in train_features.columns if c not in ignore_cols]

# Sort chronologically for out-of-time evaluation
train_features = train_features.sort_values('time_step').reset_index(drop=True)

X = train_features[feature_cols]
y = train_features['label']

tscv = TimeSeriesSplit(n_splits=5)
lgb_scores, cat_scores, xgb_scores, blend_scores = [], [], [], []

print("Running Out-of-Time Cross Validation across 5 folds...\n")

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scale_pos_weight = (len(y_tr) - sum(y_tr)) / max(sum(y_tr), 1)
    
    # 1. LightGBM (Increased to 500 estimators)
    lgb_model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.02,
        num_leaves=31,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_tr, y_tr)
    lgb_preds = lgb_model.predict_proba(X_val)[:, 1]
    
    # 2. CatBoost (Increased to 500 iterations)
    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.02,
        depth=6,
        auto_class_weights='Balanced',
        random_seed=42,
        verbose=0
    )
    cat_model.fit(X_tr, y_tr)
    cat_preds = cat_model.predict_proba(X_val)[:, 1]
    
    # 3. XGBoost
    xgb_model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc',
        tree_method='hist'
    )
    xgb_model.fit(X_tr, y_tr)
    xgb_preds = xgb_model.predict_proba(X_val)[:, 1]
    
    # 4. Re-balanced Weighted Blend (50% LGB + 40% CatBoost + 10% XGB)
    blend_preds = (0.50 * lgb_preds) + (0.40 * cat_preds) + (0.10 * xgb_preds)
    
    lgb_auc = roc_auc_score(y_val, lgb_preds)
    cat_auc = roc_auc_score(y_val, cat_preds)
    xgb_auc = roc_auc_score(y_val, xgb_preds)
    blend_auc = roc_auc_score(y_val, blend_preds)
    
    lgb_scores.append(lgb_auc)
    cat_scores.append(cat_auc)
    xgb_scores.append(xgb_auc)
    blend_scores.append(blend_auc)
    
    print(f"Fold {fold+1} | LGB: {lgb_auc:.5f} | Cat: {cat_auc:.5f} | XGB: {xgb_auc:.5f} | Blend: {blend_auc:.5f}")

print(f"\nMean Ensemble CV ROC-AUC: {np.mean(blend_scores):.5f}")

Running Out-of-Time Cross Validation across 5 folds...

Fold 1 | LGB: 0.91132 | Cat: 0.89215 | XGB: 0.95037 | Blend: 0.90820
Fold 2 | LGB: 0.99852 | Cat: 0.99819 | XGB: 0.99884 | Blend: 0.99836
Fold 3 | LGB: 0.99528 | Cat: 0.99606 | XGB: 0.99344 | Blend: 0.99594
Fold 4 | LGB: 0.99736 | Cat: 0.98568 | XGB: 0.99382 | Blend: 0.99518
Fold 5 | LGB: 0.99963 | Cat: 0.99928 | XGB: 0.99937 | Blend: 0.99949

Mean Ensemble CV ROC-AUC: 0.97943


In [23]:
scale_pos_weight_full = (len(y) - sum(y)) / max(sum(y), 1)

print("Training final LightGBM model...")
final_lgb = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.02,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight_full,
    random_state=42,
    verbose=-1
)
final_lgb.fit(X, y)

print("Training final CatBoost model...")
final_cat = CatBoostClassifier(
    iterations=500,
    learning_rate=0.02,
    depth=6,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=0
)
final_cat.fit(X, y)

print("Training final XGBoost model...")
final_xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_full,
    random_state=42,
    eval_metric='auc',
    tree_method='hist'
)
final_xgb.fit(X, y)

# Predict test set probabilities
lgb_test_preds = final_lgb.predict_proba(test_features[feature_cols])[:, 1]
cat_test_preds = final_cat.predict_proba(test_features[feature_cols])[:, 1]
xgb_test_preds = final_xgb.predict_proba(test_features[feature_cols])[:, 1]

# Re-balanced blend (50% LGB + 40% CatBoost + 10% XGB)
final_test_preds = (0.50 * lgb_test_preds) + (0.40 * cat_test_preds) + (0.10 * xgb_test_preds)

# Format & save submission
submission = pd.DataFrame({
    'index': test_df['index'],
    'target': final_test_preds
})

submission.to_csv('submission.csv', index=False)
print("Updated submission.csv successfully generated and saved!")

Training final LightGBM model...
Training final CatBoost model...
Training final XGBoost model...
Updated submission.csv successfully generated and saved!


In [24]:
import pandas as pd

sub = pd.read_csv('submission.csv')

# Verify shape, columns, missing values, and probability range
print("Shape:", sub.shape)
print("Columns:", list(sub.columns))
print("Null values:", sub.isnull().sum().sum())
print("Target Range:", sub['target'].min(), "-", sub['target'].max())
print("\nFirst 5 rows:")
print(sub.head())

Shape: (15329, 2)
Columns: ['index', 'target']
Null values: 0
Target Range: 7.784273285175993e-05 - 0.9993051092528796

First 5 rows:
   index    target
0      0  0.001172
1      1  0.007196
2      2  0.002936
3      3  0.000841
4      4  0.002038
